In [69]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.stats.diagnostic as smd
import statsmodels.stats.stattools as stattools
from statsmodels.stats.outliers_influence import variance_inflation_factor
from ipywidgets import interact, IntSlider
from statsmodels.tsa.stattools import adfuller

In [70]:
df = pd.read_csv("master.csv")
df["Date"] = pd.to_datetime(df["Date"], utc=True)

## Exploratory data analysis

### check volatility during higher attention

If true > false, we have higher volatility, meaning it is useful to look at Musk's online activity.

In [71]:
threshold = df['msk_edits'].quantile(0.90)
df['extreme_attention'] = df['msk_edits'] >= threshold

df['abs_return'] = df['log_return'].abs()

print("Average volatility during 'high' and 'low' attention days")
print(df.groupby('extreme_attention')['abs_return'].mean())

Average volatility during 'high' and 'low' attention days
extreme_attention
False    0.026761
True     0.033690
Name: abs_return, dtype: float64


## Gauss-Markov assumptions + normality + stationarity

### model

In [72]:
formula = "log_return ~ mkt_return + msk_tweets + msk_edits + tsla_edits + msk_wiki_views + tsla_wiki_views"
lm_diagnostic = smf.ols(formula, data=df).fit()
residuals = lm_diagnostic.resid
exog = lm_diagnostic.model.exog


### No perfect collinearity assumption

In [73]:
print("Variance Inflation Factor (VIF)")
vif_data = pd.DataFrame()
vif_data["Feature"] = lm_diagnostic.model.exog_names
vif_data["VIF"] = [variance_inflation_factor(exog, i) for i in range(exog.shape[1])]
print(vif_data.to_string(index=False))

Variance Inflation Factor (VIF)
        Feature      VIF
      Intercept 9.149826
     mkt_return 1.000534
     msk_tweets 1.057634
      msk_edits 1.244120
     tsla_edits 1.042765
 msk_wiki_views 2.291837
tsla_wiki_views 2.062624


### Zero conditional mean (strict exogeneity) assumption

In [74]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# creating the "Lead" variables (shift data backwards by 1 day)
df['lead_msk_edits'] = df['msk_edits'].shift(-1)
df['lead_msk_tweets'] = df['msk_tweets'].shift(-1)
df_test = df.dropna(subset=['lead_msk_edits', 'lead_msk_tweets']).copy()


# include current variables and tomorrows variables
formula_sims = """
    log_return ~ mkt_return + msk_tweets + msk_edits + 
    tsla_edits + lead_msk_edits + lead_msk_tweets
"""

model_sims = smf.ols(formula_sims, data=df_test).fit(cov_type='HC1')
print("Sims Test for Strict Exogeneity")
print(model_sims.summary().tables[1])

# joint Wald test on the lead variables
# H0: Tomorrows edits and tweets do not explain todays returns
joint_hypothesis = "(lead_msk_edits = 0), (lead_msk_tweets = 0)"
wald_sims = model_sims.wald_test(joint_hypothesis)

print("\nWald Test on Lead Variables")
print(wald_sims)
if wald_sims.pvalue < 0.05:
    print("\nResult: Reject H0. Strict exogeneity is violated.")
else:
    print("\nResult: Fail to reject H0. No evidence against strict exogeneity.")

Sims Test for Strict Exogeneity
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           0.0015      0.001      1.381      0.167      -0.001       0.004
mkt_return          1.6135      0.090     17.836      0.000       1.436       1.791
msk_tweets      -4.537e-05   5.16e-05     -0.880      0.379      -0.000    5.57e-05
msk_edits          -0.0003      0.000     -2.508      0.012      -0.001    -6.9e-05
tsla_edits          0.0001      0.000      0.774      0.439      -0.000       0.001
lead_msk_edits   9.791e-05      0.000      0.794      0.427      -0.000       0.000
lead_msk_tweets  2.872e-05   5.12e-05      0.561      0.575   -7.16e-05       0.000

Wald Test on Lead Variables
<Wald test (chi2): statistic=[[0.92990729]], p-value=0.6281642234453482, df_denom=2>

Result: Fail to reject H0. No evidence against strict exogeneity.


C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(


### No serial correlation assumprion

In [75]:
print("Breusch-Godfrey Test")
bg_stat, bg_pval, _, _ = smd.acorr_breusch_godfrey(lm_diagnostic, nlags=5)
print(f"Breusch-Godfrey (5 lags) p-value: {bg_pval:.4e}")
if bg_pval < 0.05:
    print("Result: Reject H0. Serial correlation is present (Violates Gauss-Markov).")
else:
    print("Result: Fail to reject H0. No significant serial correlation detected.")

Breusch-Godfrey Test
Breusch-Godfrey (5 lags) p-value: 6.9264e-01
Result: Fail to reject H0. No significant serial correlation detected.


### Homoscedasticity assumption

In [76]:
print("Breusch-Pagan Test")
bp_stat, bp_pval, _, _ = smd.het_breuschpagan(residuals, exog)
print(f"Breusch-Pagan p-value: {bp_pval:.4e}")
if bp_pval < 0.05:
    print("Result: Reject H0. Heteroscedasticity is present.")
else:
    print("Result: Fail to reject H0. Homoscedasticity holds.")

Breusch-Pagan Test
Breusch-Pagan p-value: 1.4790e-13
Result: Reject H0. Heteroscedasticity is present.


we should use HC1 errors

### Normality assumption

In [77]:
print("Jarque-Bera Test")
jb_stat, jb_pval, skew, kurtosis = stattools.jarque_bera(residuals)
print(f"Jarque-Bera p-value: {jb_pval:.4e}")
print(f"Skewness: {skew:.4f}, Kurtosis: {kurtosis:.4f}")
if jb_pval < 0.05:
    print("Result: Reject H0. Residuals are not normally distributed.")
else:
    print("Result: Fail to reject H0. Residuals are normally distributed.")

Jarque-Bera Test
Jarque-Bera p-value: 0.0000e+00
Skewness: 0.0705, Kurtosis: 7.7057
Result: Reject H0. Residuals are not normally distributed.


### Stationarity

In [79]:
variables_to_test = [
    'log_return', 
    'mkt_return', 
    'msk_tweets', 
    'msk_edits', 
    'tsla_edits'
]

print("Augmented Dickey-Fuller (ADF) Stationarity Tests")
for var in variables_to_test:
    adf_result = adfuller(df[var], autolag='AIC')
    adf_stat = adf_result[0]
    p_value = adf_result[1]
    
    print(f"\nVariable: {var}")
    print(f"ADF Statistic: {adf_stat:.4f}")
    print(f"p-value:       {p_value:.4f}")
    
    if p_value < 0.05:
        print("Result: Reject H0. The series is STATIONARY")
    else:
        print("Result: Fail to reject H0. The series is NON-STATIONARY")

Augmented Dickey-Fuller (ADF) Stationarity Tests

Variable: log_return
ADF Statistic: -46.1344
p-value:       0.0000
Result: Reject H0. The series is STATIONARY

Variable: mkt_return
ADF Statistic: -14.0319
p-value:       0.0000
Result: Reject H0. The series is STATIONARY

Variable: msk_tweets
ADF Statistic: -1.3445
p-value:       0.6086
Result: Fail to reject H0. The series is NON-STATIONARY

Variable: msk_edits
ADF Statistic: -5.7462
p-value:       0.0000
Result: Reject H0. The series is STATIONARY

Variable: tsla_edits
ADF Statistic: -11.0626
p-value:       0.0000
Result: Reject H0. The series is STATIONARY
